# Series 2 — Post 2: Event Time, Windows, Watermarks, and Late Data

## Objective

Build a controlled order-event stream to understand:

- Event time versus processing time
- Tumbling, sliding, and session windows
- Late-arriving events
- Watermark thresholds
- Stateful processing
- Window finalization and output modes

## Core question

How long should a streaming pipeline wait for late data before finalizing a result?

A longer watermark can improve completeness, but it also keeps more state.

A shorter watermark reduces state and latency, but very late events may not update finalized results.

In [0]:
for active_query in spark.streams.active:
    print(f"Stopping: {active_query.name or active_query.id}")
    active_query.stop()

print("✅ All active streaming queries stopped")

In [0]:
print(f"Active streams: {len(spark.streams.active)}")

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.streaming_lab;

CREATE VOLUME IF NOT EXISTS
workspace.streaming_lab.streaming_volume;

USE CATALOG workspace;
USE SCHEMA streaming_lab;

In [0]:
from datetime import datetime

from pyspark.sql import Row
from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType,
)

In [0]:
catalog = "workspace"
schema = "streaming_lab"

base_path = (
    "/Volumes/workspace/streaming_lab/"
    "streaming_volume"
)

source_path = f"{base_path}/order_events"
checkpoint_base = f"{base_path}/checkpoints"

tumbling_table = (
    "workspace.streaming_lab."
    "tumbling_5m_no_watermark"
)

tumbling_checkpoint = (
    f"{checkpoint_base}/tumbling_5m_no_watermark"
)

deduplicated_table = (
    "workspace.streaming_lab."
    "order_events_deduplicated"
)

dedup_checkpoint = (
    f"{checkpoint_base}/order_events_deduplicated"
)

print(f"Source path: {source_path}")
print(f"Checkpoint base: {checkpoint_base}")
print(f"Tumbling table: {tumbling_table}")
print(f"Deduplicated table: {deduplicated_table}")

In [0]:
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("event_ts", TimestampType(), False),
])

In [0]:
catalog = "workspace"
schema = "streaming_lab"

base_path = (
    "/Volumes/workspace/streaming_lab/"
    "streaming_volume"
)

source_path = f"{base_path}/order_events"
checkpoint_base = f"{base_path}/checkpoints"

print(f"Source: {source_path}")
print(f"Checkpoints: {checkpoint_base}")

In [0]:
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType,
)

#Define Schema
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("event_ts", TimestampType(), False),
])

#Reset Source Folder before first run
dbutils.fs.rm(source_path, True)
dbutils.fs.mkdirs(source_path)

print("✅ Streaming source folder reset")

In [0]:
#Create Batch 1
batch_1 = [
    Row(
        event_id="EVT-001",
        order_id="ORD-001",
        user_id="U-001",
        event_type="purchase",
        amount=100.0,
        event_ts=datetime(2026, 7, 22, 10, 0, 0),
    ),
    Row(
        event_id="EVT-002",
        order_id="ORD-002",
        user_id="U-002",
        event_type="purchase",
        amount=80.0,
        event_ts=datetime(2026, 7, 22, 10, 2, 0),
    ),
    Row(
        event_id="EVT-003",
        order_id="ORD-003",
        user_id="U-001",
        event_type="purchase",
        amount=120.0,
        event_ts=datetime(2026, 7, 22, 10, 4, 0),
    ),
    Row(
        event_id="EVT-004",
        order_id="ORD-004",
        user_id="U-003",
        event_type="purchase",
        amount=60.0,
        event_ts=datetime(2026, 7, 22, 10, 7, 0),
    ),
]

In [0]:
#Create Batch 2
batch_2 = [
    Row(
        event_id="EVT-005",
        order_id="ORD-005",
        user_id="U-002",
        event_type="purchase",
        amount=150.0,
        event_ts=datetime(2026, 7, 22, 10, 12, 0),
    ),
    Row(
        event_id="EVT-006",
        order_id="ORD-006",
        user_id="U-001",
        event_type="purchase",
        amount=90.0,
        event_ts=datetime(2026, 7, 22, 10, 16, 0),
    ),

    # Duplicate of EVT-003
    Row(
        event_id="EVT-003",
        order_id="ORD-003",
        user_id="U-001",
        event_type="purchase",
        amount=120.0,
        event_ts=datetime(2026, 7, 22, 10, 4, 0),
    ),

    # Late relative to the events already seen, but within 10 minutes
    Row(
        event_id="EVT-007",
        order_id="ORD-007",
        user_id="U-003",
        event_type="purchase",
        amount=70.0,
        event_ts=datetime(2026, 7, 22, 10, 9, 0),
    ),
]

In [0]:
#Create Batch 3
batch_3 = [
    Row(
        event_id="EVT-008",
        order_id="ORD-008",
        user_id="U-002",
        event_type="purchase",
        amount=200.0,
        event_ts=datetime(2026, 7, 22, 10, 28, 0),
    ),

    # Late, but still within the 10-minute threshold
    Row(
        event_id="EVT-009",
        order_id="ORD-009",
        user_id="U-004",
        event_type="purchase",
        amount=55.0,
        event_ts=datetime(2026, 7, 22, 10, 20, 0),
    ),

    # Very late: 28 minutes behind the maximum event timestamp
    Row(
        event_id="EVT-010",
        order_id="ORD-010",
        user_id="U-005",
        event_type="purchase",
        amount=40.0,
        event_ts=datetime(2026, 7, 22, 10, 0, 0),
    ),
]

In [0]:
spark.sql(
    "DROP TABLE IF EXISTS "
    "workspace.streaming_lab.tumbling_5m_no_watermark"
)

spark.sql(
    "DROP TABLE IF EXISTS "
    "workspace.streaming_lab.order_events_deduplicated"
)

dbutils.fs.rm(source_path, True)
dbutils.fs.rm(checkpoint_base, True)

dbutils.fs.mkdirs(source_path)
dbutils.fs.mkdirs(checkpoint_base)

print("✅ Streaming source reset")
print("✅ Checkpoints reset")
print("✅ Output tables dropped")

In [0]:
display(dbutils.fs.ls(source_path))

In [0]:
#Write only Batch 1
batch_1_df = spark.createDataFrame(
    batch_1,
    schema=event_schema
)

(
    batch_1_df
    .coalesce(1)
    .write
    .mode("append")
    .json(f"{source_path}/batch_1")
)

display(batch_1_df.orderBy("event_ts"))

In [0]:
#Verify Source FOlder

display(dbutils.fs.ls(source_path))

In [0]:
#Create Streaming read
from pyspark.sql import functions as F

stream_df = (
    spark.readStream
    .schema(event_schema)
    .option("recursiveFileLookup", "true")
    .json(source_path)
)

print(f"Is streaming: {stream_df.isStreaming}")

In [0]:
#Create 5-minute tumbling windows
tumbling_5m_df = (
    stream_df
    .groupBy(
        F.window(
            F.col("event_ts"),
            "5 minutes"
        )
    )
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount").alias("total_amount"),
        F.approx_count_distinct("user_id").alias("unique_users")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "event_count",
        F.round("total_amount", 2).alias("total_amount"),
        "unique_users"
    )
)

In [0]:
#Write result with foreachBatch
tumbling_table = (
    "workspace.streaming_lab."
    "tumbling_5m_no_watermark"
)

tumbling_checkpoint = (
    f"{checkpoint_base}/tumbling_5m_no_watermark"
)

#dbutils.fs.rm(tumbling_checkpoint, True)

def overwrite_tumbling_result(batch_df, batch_id):
    print(
        f"Processing batch_id={batch_id}, "
        f"output rows={batch_df.count()}"
    )

    (
        batch_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tumbling_table)
    )

query = (
    tumbling_5m_df
    .writeStream
    .outputMode("complete")
    .foreachBatch(overwrite_tumbling_result)
    .option(
        "checkpointLocation",
        tumbling_checkpoint
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("✅ Batch 1 processed")

In [0]:
%sql

SHOW TABLES IN workspace.streaming_lab LIKE 'tumbling_5m_no_watermark';

--Inspect Batch 1 results
SELECT *
FROM workspace.streaming_lab.tumbling_5m_no_watermark
ORDER BY window_start;

In [0]:
result_df = spark.table(tumbling_table)

assert result_df.count() == 2, (
    f"Expected 2 windows, found {result_df.count()}"
)

first_window = (
    result_df
    .filter(
        F.col("window_start")
        == F.to_timestamp(
            F.lit("2026-07-22 10:00:00")
        )
    )
    .first()
)

assert first_window is not None, (
    "10:00–10:05 window was not created"
)

assert first_window["event_count"] == 3, (
    f"Expected 3 events, "
    f"found {first_window['event_count']}"
)

assert first_window["total_amount"] == 300.0, (
    f"Expected total 300, "
    f"found {first_window['total_amount']}"
)

print("✅ Two tumbling windows created")
print("✅ First window contains 3 events")
print("✅ First window total is 300")
print("✅ Baseline without watermark passed")

In [0]:
#add Batch 2
batch_2_df = spark.createDataFrame(
    batch_2,
    schema=event_schema
)

(
    batch_2_df
    .coalesce(1)
    .write
    .mode("append")
    .json(f"{source_path}/batch_2")
)

display(batch_2_df.orderBy("event_ts"))

In [0]:
display(dbutils.fs.ls(source_path))

In [0]:
#rereun same batch query

query = (
    tumbling_5m_df
    .writeStream
    .outputMode("complete")
    .foreachBatch(overwrite_tumbling_result)
    .option(
        "checkpointLocation",
        tumbling_checkpoint
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("✅ Batch 2 processed using the existing checkpoint")

In [0]:
%sql

SELECT *
FROM workspace.streaming_lab.tumbling_5m_no_watermark
ORDER BY window_start;

In [0]:
#Validate Batch 2 Behavior

result_df = spark.table(tumbling_table)

first_window = (
    result_df
    .filter(
        F.col("window_start")
        == F.to_timestamp(
            F.lit("2026-07-22 10:00:00")
        )
    )
    .first()
)

late_window = (
    result_df
    .filter(
        F.col("window_start")
        == F.to_timestamp(
            F.lit("2026-07-22 10:05:00")
        )
    )
    .first()
)

assert result_df.count() == 4, (
    f"Expected 4 windows, found {result_df.count()}"
)

assert first_window["event_count"] == 4, (
    "Duplicate EVT-003 was not reflected "
    "in the no-watermark baseline"
)

assert first_window["total_amount"] == 420.0, (
    f"Expected 420 after duplicate, "
    f"found {first_window['total_amount']}"
)

assert late_window["event_count"] == 2, (
    "Late EVT-007 did not update its historical window"
)

assert late_window["total_amount"] == 130.0, (
    f"Expected 130 after late event, "
    f"found {late_window['total_amount']}"
)

print("✅ Existing checkpoint processed only Batch 2")
print("✅ Duplicate event increased the first window")
print("✅ Late event updated an older window")
print("✅ No-watermark behavior confirmed")

## Watermark and deduplication

The raw source can contain:

- Duplicate event IDs
- Events that arrive later than newer events
- Events that arrive beyond the accepted lateness threshold

This stage applies a 10-minute event-time watermark and retains only one record per event ID.

In [0]:
spark.sql(
    "DROP TABLE IF EXISTS "
    "workspace.streaming_lab.order_events_deduplicated"
)

dbutils.fs.rm(
    dedup_checkpoint,
    True
)

print("✅ Dedup table and checkpoint reset")

In [0]:
#Create the deduplication stream
deduplicated_stream_df = (
    spark.readStream
    .schema(event_schema)
    .option("recursiveFileLookup", "true")
    .json(source_path)
    .withWatermark(
        "event_ts",
        "10 minutes"
    )
    .dropDuplicatesWithinWatermark(
        ["event_id"]
    )
)

In [0]:
#Write deduplicated events to Delta
dedup_query = (
    deduplicated_stream_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        dedup_checkpoint
    )
    .trigger(availableNow=True)
    .toTable(deduplicated_table)
)

dedup_query.awaitTermination()

print("✅ Batches 1 and 2 deduplicated")

In [0]:
%sql
SELECT
    event_id,
    order_id,
    user_id,
    amount,
    event_ts
FROM workspace.streaming_lab.order_events_deduplicated
ORDER BY event_ts, event_id;

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT event_id) AS distinct_event_count
FROM workspace.streaming_lab.order_events_deduplicated;

In [0]:
#write Batch 3

batch_3_df = spark.createDataFrame(
    batch_3,
    schema=event_schema
)

(
    batch_3_df
    .coalesce(1)
    .write
    .mode("append")
    .json(f"{source_path}/batch_3")
)

display(batch_3_df.orderBy("event_ts"))

In [0]:
display(
    dbutils.fs.ls(source_path)
)

In [0]:
#Process Batch 3 using the existing dedup checkpoint
dedup_query = (
    deduplicated_stream_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        dedup_checkpoint
    )
    .trigger(availableNow=True)
    .toTable(deduplicated_table)
)

dedup_query.awaitTermination()

print("✅ Batch 3 processed")

In [0]:
%sql
SELECT
    event_id,
    amount,
    event_ts
FROM workspace.streaming_lab.order_events_deduplicated
ORDER BY event_ts, event_id;

In [0]:
%sql
SELECT
    event_id,
    event_ts
FROM workspace.streaming_lab.order_events_deduplicated
WHERE event_id IN (
    'EVT-008',
    'EVT-009',
    'EVT-010'
)
ORDER BY event_id;

In [0]:
#Validate Final Result
final_dedup_df = spark.table(
    deduplicated_table
)

final_count = final_dedup_df.count()

duplicate_event_ids = (
    final_dedup_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

evt_003_count = (
    final_dedup_df
    .filter(F.col("event_id") == "EVT-003")
    .count()
)

evt_008_count = (
    final_dedup_df
    .filter(F.col("event_id") == "EVT-008")
    .count()
)

evt_009_count = (
    final_dedup_df
    .filter(F.col("event_id") == "EVT-009")
    .count()
)

evt_010_count = (
    final_dedup_df
    .filter(F.col("event_id") == "EVT-010")
    .count()
)

print(f"Final count: {final_count}")
print(f"Duplicate event IDs: {duplicate_event_ids}")
print(f"EVT-003 count: {evt_003_count}")
print(f"EVT-008 count: {evt_008_count}")
print(f"EVT-009 count: {evt_009_count}")
print(f"EVT-010 count: {evt_010_count}")

In [0]:
assert final_count == 9, (
    f"Expected 9 events, found {final_count}"
)

assert duplicate_event_ids == 0, (
    f"Found {duplicate_event_ids} duplicate event IDs"
)

assert evt_003_count == 1, (
    "EVT-003 should appear exactly once"
)

assert evt_008_count == 1, (
    "EVT-008 should be accepted"
)

assert evt_009_count == 1, (
    "EVT-009 should be accepted"
)

assert evt_010_count == 0, (
    "EVT-010 should be dropped as very late"
)

print("✅ Environment rebuilt")
print("✅ No-watermark baseline restored")
print("✅ Duplicate EVT-003 removed")
print("✅ Late EVT-009 accepted")
print("✅ Very-late EVT-010 dropped")
print("✅ Ready for tumbling, sliding, and session windows")

In [0]:
#Define window-query objects
dedup_source_table = (
    "workspace.streaming_lab."
    "order_events_deduplicated"
)

tumbling_watermark_table = (
    "workspace.streaming_lab."
    "tumbling_5m_deduplicated"
)

sliding_window_table = (
    "workspace.streaming_lab."
    "sliding_10m_every_5m"
)

window_checkpoint_base = (
    f"{checkpoint_base}/window_comparison"
)

tumbling_watermark_checkpoint = (
    f"{window_checkpoint_base}/tumbling_5m"
)

sliding_window_checkpoint = (
    f"{window_checkpoint_base}/sliding_10m_every_5m"
)

In [0]:
#Reset only new window outputs
spark.sql(
    f"DROP TABLE IF EXISTS {tumbling_watermark_table}"
)

spark.sql(
    f"DROP TABLE IF EXISTS {sliding_window_table}"
)

dbutils.fs.rm(
    window_checkpoint_base,
    True
)

print("✅ Window comparison outputs reset")

In [0]:
#Read deduplicated delta table as stream
dedup_delta_stream_df = (
    spark.readStream
    .table(dedup_source_table)
)

print(
    f"Streaming source: "
    f"{dedup_delta_stream_df.isStreaming}"
)

In [0]:
#5-minute tumbling windows aggregation
tumbling_5m_dedup_df = (
    dedup_delta_stream_df
    .withWatermark(
        "event_ts",
        "10 minutes"
    )
    .groupBy(
        F.window(
            F.col("event_ts"),
            "5 minutes"
        )
    )
    .agg(
        F.count("*").alias("event_count"),
        F.round(
            F.sum("amount"),
            2
        ).alias("total_amount"),
        F.approx_count_distinct(
            "user_id"
        ).alias("unique_users")
    )
    .select(
        F.col("window.start").alias(
            "window_start"
        ),
        F.col("window.end").alias(
            "window_end"
        ),
        "event_count",
        "total_amount",
        "unique_users"
    )
)

In [0]:
#Define a reusable complete result writer
def overwrite_complete_result(
    batch_df,
    batch_id,
    target_table
):
    row_count = batch_df.count()

    print(
        f"batch_id={batch_id}, "
        f"rows={row_count}, "
        f"target={target_table}"
    )

    (
        batch_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(target_table)
    )

In [0]:
#run tumbling window stream
tumbling_query = (
    tumbling_5m_dedup_df
    .writeStream
    .outputMode("complete")
    .foreachBatch(
        lambda batch_df, batch_id:
            overwrite_complete_result(
                batch_df,
                batch_id,
                tumbling_watermark_table
            )
    )
    .option(
        "checkpointLocation",
        tumbling_watermark_checkpoint
    )
    .trigger(
        availableNow=True
    )
    .start()
)

tumbling_query.awaitTermination()

print("✅ Tumbling-window query completed")

In [0]:
%sql
--Inspect tumbling windows
SELECT
    window_start,
    window_end,
    event_count,
    total_amount,
    unique_users
FROM workspace.streaming_lab.tumbling_5m_deduplicated
ORDER BY window_start;

In [0]:
#Validate tumbling behavior
tumbling_result_df = spark.table(
    tumbling_watermark_table
)

tumbling_window_count = (
    tumbling_result_df.count()
)

tumbling_event_total = (
    tumbling_result_df
    .agg(
        F.sum("event_count").alias(
            "total_events"
        )
    )
    .first()["total_events"]
)

tumbling_amount_total = (
    tumbling_result_df
    .agg(
        F.sum("total_amount").alias(
            "total_amount"
        )
    )
    .first()["total_amount"]
)

assert tumbling_window_count == 6, (
    f"Expected 6 windows, "
    f"found {tumbling_window_count}"
)

assert tumbling_event_total == 9, (
    f"Expected 9 event assignments, "
    f"found {tumbling_event_total}"
)

assert tumbling_amount_total == 925.0, (
    f"Expected total amount 925, "
    f"found {tumbling_amount_total}"
)

print("✅ Six tumbling windows created")
print("✅ Every event appears in one window")
print("✅ Nine event assignments")
print("✅ Total amount remains 925")

In [0]:
#ten-minute sliding windows
#Create sliding-window aggregation
sliding_10m_df = (
    dedup_delta_stream_df
    .withWatermark(
        "event_ts",
        "10 minutes"
    )
    .groupBy(
        F.window(
            F.col("event_ts"),
            "10 minutes",
            "5 minutes"
        )
    )
    .agg(
        F.count("*").alias("event_count"),
        F.round(
            F.sum("amount"),
            2
        ).alias("total_amount"),
        F.approx_count_distinct(
            "user_id"
        ).alias("unique_users")
    )
    .select(
        F.col("window.start").alias(
            "window_start"
        ),
        F.col("window.end").alias(
            "window_end"
        ),
        "event_count",
        "total_amount",
        "unique_users"
    )
)

In [0]:
#run sliding-window stream
sliding_query = (
    sliding_10m_df
    .writeStream
    .outputMode("complete")
    .foreachBatch(
        lambda batch_df, batch_id:
            overwrite_complete_result(
                batch_df,
                batch_id,
                sliding_window_table
            )
    )
    .option(
        "checkpointLocation",
        sliding_window_checkpoint
    )
    .trigger(
        availableNow=True
    )
    .start()
)

sliding_query.awaitTermination()

print("✅ Sliding-window query completed")

In [0]:
%sql
--Inspect sliding windows
SELECT
    window_start,
    window_end,
    event_count,
    total_amount,
    unique_users
FROM workspace.streaming_lab.sliding_10m_every_5m
ORDER BY window_start;

In [0]:
%sql
--comparing tumbling and sliding window behavior
SELECT
    'TUMBLING' AS window_type,
    COUNT(*) AS number_of_windows,
    SUM(event_count) AS event_assignments,
    SUM(total_amount) AS summed_window_amount
FROM workspace.streaming_lab.tumbling_5m_deduplicated

UNION ALL

SELECT
    'SLIDING' AS window_type,
    COUNT(*) AS number_of_windows,
    SUM(event_count) AS event_assignments,
    SUM(total_amount) AS summed_window_amount
FROM workspace.streaming_lab.sliding_10m_every_5m;

In [0]:
#Validate the comparison
sliding_result_df = spark.table(
    sliding_window_table
)

sliding_window_count = (
    sliding_result_df.count()
)

sliding_event_assignments = (
    sliding_result_df
    .agg(
        F.sum("event_count").alias(
            "event_assignments"
        )
    )
    .first()["event_assignments"]
)

sliding_amount_total = (
    sliding_result_df
    .agg(
        F.sum("total_amount").alias(
            "summed_amount"
        )
    )
    .first()["summed_amount"]
)

assert sliding_window_count > 6, (
    "Sliding windows should create more "
    "intervals than tumbling windows"
)

assert sliding_event_assignments > 9, (
    "Sliding windows should assign some "
    "events to multiple windows"
)

assert sliding_amount_total > 925.0, (
    "Overlapping windows should produce a "
    "summed window amount above the raw total"
)

print("✅ Sliding windows overlap")
print(
    f"✅ Sliding window count: "
    f"{sliding_window_count}"
)
print(
    f"✅ Event assignments: "
    f"{sliding_event_assignments}"
)
print(
    f"✅ Summed window amount: "
    f"{sliding_amount_total}"
)
print("✅ Tumbling vs sliding comparison passed")

In [0]:
#Define session window objects

session_window_table = (
    "workspace.streaming_lab."
    "session_5m_by_user"
)

session_window_checkpoint = (
    f"{window_checkpoint_base}/session_5m_by_user"
)

In [0]:
#read only new output
spark.sql(
    f"DROP TABLE IF EXISTS {session_window_table}"
)

dbutils.fs.rm(
    session_window_checkpoint,
    True
)

print("✅ Session-window output reset")

In [0]:
#Create session window aggregation
session_5m_df = (
    dedup_delta_stream_df
    .withWatermark(
        "event_ts",
        "10 minutes"
    )
    .groupBy(
        "user_id",
        F.session_window(
            F.col("event_ts"),
            "5 minutes"
        )
    )
    .agg(
        F.count("*").alias("event_count"),
        F.round(
            F.sum("amount"),
            2
        ).alias("total_amount"),
        F.min("event_ts").alias(
            "first_event_ts"
        ),
        F.max("event_ts").alias(
            "last_event_ts"
        )
    )
    .select(
        "user_id",
        F.col("session_window.start").alias(
            "session_start"
        ),
        F.col("session_window.end").alias(
            "session_end"
        ),
        "first_event_ts",
        "last_event_ts",
        "event_count",
        "total_amount"
    )
)

In [0]:
#run the session window stream
session_query = (
    session_5m_df
    .writeStream
    .outputMode("complete")
    .foreachBatch(
        lambda batch_df, batch_id:
            overwrite_complete_result(
                batch_df,
                batch_id,
                session_window_table
            )
    )
    .option(
        "checkpointLocation",
        session_window_checkpoint
    )
    .trigger(
        availableNow=True
    )
    .start()
)

session_query.awaitTermination()

print("✅ Session-window query completed")

In [0]:
%sql
--Inspect session results
SELECT
    user_id,
    session_start,
    session_end,
    first_event_ts,
    last_event_ts,
    event_count,
    total_amount
FROM workspace.streaming_lab.session_5m_by_user
ORDER BY user_id, session_start;

In [0]:
#Validate Sessions
session_result_df = spark.table(
    session_window_table
)

u001_sessions = (
    session_result_df
    .filter(F.col("user_id") == "U-001")
    .orderBy("session_start")
    .collect()
)

u002_session_count = (
    session_result_df
    .filter(F.col("user_id") == "U-002")
    .count()
)

u003_sessions = (
    session_result_df
    .filter(F.col("user_id") == "U-003")
    .collect()
)

assert len(u001_sessions) == 2, (
    f"Expected 2 sessions for U-001, "
    f"found {len(u001_sessions)}"
)

assert u001_sessions[0]["event_count"] == 2, (
    "The first U-001 session should contain "
    "the 10:00 and 10:04 events"
)

assert u001_sessions[0]["total_amount"] == 220.0, (
    f"Expected first U-001 session total 220, "
    f"found {u001_sessions[0]['total_amount']}"
)

assert u001_sessions[1]["event_count"] == 1, (
    "The second U-001 session should contain "
    "only the 10:16 event"
)

assert u002_session_count == 3, (
    f"Expected 3 sessions for U-002, "
    f"found {u002_session_count}"
)

assert len(u003_sessions) == 1, (
    f"Expected 1 session for U-003, "
    f"found {len(u003_sessions)}"
)

assert u003_sessions[0]["event_count"] == 2, (
    "The U-003 session should contain "
    "the 10:07 and 10:09 events"
)

assert u003_sessions[0]["total_amount"] == 130.0, (
    f"Expected U-003 session total 130, "
    f"found {u003_sessions[0]['total_amount']}"
)

print("✅ U-001 created two sessions")
print("✅ U-001 first session contains two events")
print("✅ U-002 created three separate sessions")
print("✅ U-003 events merged into one session")
print("✅ Session-window validation passed")

In [0]:
%sql
--Compare all windows
SELECT
    'TUMBLING' AS window_type,
    COUNT(*) AS result_rows,
    SUM(event_count) AS event_assignments,
    'Fixed, non-overlapping clock intervals' AS interpretation
FROM workspace.streaming_lab.tumbling_5m_deduplicated

UNION ALL

SELECT
    'SLIDING' AS window_type,
    COUNT(*) AS result_rows,
    SUM(event_count) AS event_assignments,
    'Overlapping rolling intervals' AS interpretation
FROM workspace.streaming_lab.sliding_10m_every_5m

UNION ALL

SELECT
    'SESSION' AS window_type,
    COUNT(*) AS result_rows,
    SUM(event_count) AS event_assignments,
    'Activity groups separated by inactivity' AS interpretation
FROM workspace.streaming_lab.session_5m_by_user;

## Tumbling vs sliding vs session windows

### Tumbling windows

Fixed and non-overlapping.

Use for additive reporting such as revenue every five minutes, hourly order totals, or daily transaction counts.

### Sliding windows

Fixed length but overlapping.

Use for rolling trends, fraud detection, traffic monitoring, or alerts such as “orders during the last ten minutes.”

Do not sum monetary values across all sliding-window rows because the same event can belong to multiple windows.

### Session windows

Dynamic boundaries based on inactivity.

Use for customer visits, application usage, device activity, support conversations, or user journeys.

A new event extends the current session when it arrives before the inactivity gap expires. Otherwise, it starts a new session.

In [0]:
#Inspect query progress
import json

dedup_progress = dedup_query.lastProgress

print(
    json.dumps(
        dedup_progress,
        indent=2,
        default=str
    )
)

In [0]:
#Extract important deduplication metrics
def print_streaming_metrics(
    query,
    query_name
):
    progress = query.lastProgress

    if not progress:
        print(
            f"No progress found for {query_name}"
        )
        return

    print(f"Query: {query_name}")
    print(
        f"Batch ID: "
        f"{progress.get('batchId')}"
    )
    print(
        f"Input rows: "
        f"{progress.get('numInputRows', 0)}"
    )
    print(
        f"Input rows/sec: "
        f"{progress.get('inputRowsPerSecond', 0)}"
    )
    print(
        f"Processed rows/sec: "
        f"{progress.get('processedRowsPerSecond', 0)}"
    )

    event_time = progress.get(
        "eventTime",
        {}
    )

    print(
        f"Maximum event time: "
        f"{event_time.get('max')}"
    )
    print(
        f"Watermark: "
        f"{event_time.get('watermark')}"
    )

    state_operators = progress.get(
        "stateOperators",
        []
    )

    if not state_operators:
        print(
            "No stateful operators reported"
        )
        return

    for index, operator in enumerate(
        state_operators
    ):
        print(
            f"\nState operator {index}"
        )
        print(
            f"Name: "
            f"{operator.get('operatorName')}"
        )
        print(
            f"Total state rows: "
            f"{operator.get('numRowsTotal', 0)}"
        )
        print(
            f"Updated state rows: "
            f"{operator.get('numRowsUpdated', 0)}"
        )
        print(
            f"Removed state rows: "
            f"{operator.get('numRowsRemoved', 0)}"
        )
        print(
            f"Dropped by watermark: "
            f"{operator.get('numRowsDroppedByWatermark', 0)}"
        )
        print(
            f"State memory bytes: "
            f"{operator.get('memoryUsedBytes', 0)}"
        )

In [0]:
print_streaming_metrics(
    dedup_query,
    "Watermark deduplication"
)

In [0]:
#Inspect all recent micro-batches
dedup_recent_progress = (
    dedup_query.recentProgress
)

for progress in dedup_recent_progress:
    print(
        "Batch:",
        progress.get("batchId"),
        "| Input:",
        progress.get("numInputRows"),
        "| Watermark:",
        progress.get(
            "eventTime",
            {}
        ).get("watermark"),
    )

    for operator in progress.get(
        "stateOperators",
        []
    ):
        print(
            "  State rows:",
            operator.get(
                "numRowsTotal",
                0
            ),
            "| Updated:",
            operator.get(
                "numRowsUpdated",
                0
            ),
            "| Dropped late:",
            operator.get(
                "numRowsDroppedByWatermark",
                0
            ),
        )

In [0]:
#Build a reusable metrics table
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
)

def safe_int(value, default=0):
    if value is None:
        return default

    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def safe_str(value, default="N/A"):
    if value is None:
        return default

    return str(value)


progress_rows = []

for progress in dedup_query.recentProgress:
    if not progress:
        continue

    state_operators = progress.get("stateOperators") or []
    state = state_operators[0] if state_operators else {}

    event_time = progress.get("eventTime") or {}

    progress_rows.append({
        "batch_id": safe_int(
            progress.get("batchId")
        ),
        "input_rows": safe_int(
            progress.get("numInputRows")
        ),
        "watermark": safe_str(
            event_time.get("watermark")
        ),
        "maximum_event_time": safe_str(
            event_time.get("max")
        ),
        "state_rows_total": safe_int(
            state.get("numRowsTotal")
        ),
        "state_rows_updated": safe_int(
            state.get("numRowsUpdated")
        ),
        "state_rows_removed": safe_int(
            state.get("numRowsRemoved")
        ),
        "rows_dropped_by_watermark": safe_int(
            state.get("numRowsDroppedByWatermark")
        ),
        "state_memory_bytes": safe_int(
            state.get("memoryUsedBytes")
        ),
    })


progress_schema = StructType([
    StructField("batch_id", LongType(), False),
    StructField("input_rows", LongType(), False),
    StructField("watermark", StringType(), False),
    StructField("maximum_event_time", StringType(), False),
    StructField("state_rows_total", LongType(), False),
    StructField("state_rows_updated", LongType(), False),
    StructField("state_rows_removed", LongType(), False),
    StructField(
        "rows_dropped_by_watermark",
        LongType(),
        False,
    ),
    StructField("state_memory_bytes", LongType(), False),
])


if not progress_rows:
    print("No recent progress records found for dedup_query")
else:
    progress_metrics_df = spark.createDataFrame(
        progress_rows,
        schema=progress_schema,
    )

    display(
        progress_metrics_df.orderBy("batch_id")
    )

In [0]:
#Demonstrate apped mode finalization
append_window_table = (
    "workspace.streaming_lab."
    "tumbling_5m_append"
)

append_window_checkpoint = (
    f"{window_checkpoint_base}/"
    "tumbling_5m_append"
)

In [0]:
spark.sql(
    f"DROP TABLE IF EXISTS "
    f"{append_window_table}"
)

dbutils.fs.rm(
    append_window_checkpoint,
    True
)

print("✅ Append-mode output reset")

In [0]:
append_tumbling_df = (
    dedup_delta_stream_df
    .withWatermark(
        "event_ts",
        "10 minutes"
    )
    .groupBy(
        F.window(
            F.col("event_ts"),
            "5 minutes"
        )
    )
    .agg(
        F.count("*").alias(
            "event_count"
        ),
        F.round(
            F.sum("amount"),
            2
        ).alias(
            "total_amount"
        )
    )
    .select(
        F.col("window.start").alias(
            "window_start"
        ),
        F.col("window.end").alias(
            "window_end"
        ),
        "event_count",
        "total_amount"
    )
)

In [0]:
append_query = (
    append_tumbling_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        append_window_checkpoint
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        append_window_table
    )
)

append_query.awaitTermination()

print("✅ Append-mode query completed")

In [0]:
%sql
SELECT *
FROM workspace.streaming_lab.tumbling_5m_append
ORDER BY window_start;

In [0]:
%sql
--Compare complete and append outputs
SELECT
    'COMPLETE' AS output_mode,
    COUNT(*) AS emitted_windows,
    MIN(window_start) AS first_window,
    MAX(window_start) AS last_window
FROM workspace.streaming_lab.tumbling_5m_deduplicated

UNION ALL

SELECT
    'APPEND' AS output_mode,
    COUNT(*) AS emitted_windows,
    MIN(window_start) AS first_window,
    MAX(window_start) AS last_window
FROM workspace.streaming_lab.tumbling_5m_append;

## Stateful processing and output modes

A watermark serves two related purposes:

1. It defines how late an event may arrive and still update a stateful result.
2. It allows Spark to eventually remove old state and finalize windows.

### Complete mode

Emits the entire aggregation result during every trigger.

Useful for small tests, but expensive for large production state.

### Append mode

Emits only finalized windows after the watermark passes their end time.

Useful for stable downstream records, but introduces intentional latency.

### Update mode

Emits only rows changed during the current trigger.

Useful for continuously refreshed results, often combined with `foreachBatch` and a Delta `MERGE`.

The correct mode depends on whether the consumer needs:

- Immediate provisional results
- Changed results
- Or finalized immutable results